In [ ]:
import os
from pathlib import Path

def root_dir(marker="EXPLib"):
    cwd = Path.cwd()
    # if cwd already contains the marker
    if (cwd / marker).is_dir():
        return cwd
    # otherwise walk up the parents
    for p in cwd.parents:
        if (p / marker).is_dir():
            return p
    raise FileNotFoundError(f"Root marker '{marker}' not found")

os.chdir(root_dir())
#print(os.getcwd())

In [ ]:
from EXPLib.xai_library.supervisor.surrogate_model.toymodel_symbolic_constraint_utils import create_dataloaders, compute_bbox_constraints_from_coco_json, create_ssdlite, train_one_epoch

import torch
from torch.optim import SGD
from torch.optim.lr_scheduler import StepLR
from torch.amp import GradScaler

In [ ]:
params = {
    "batch_size": 16,
    "dataset_path":  './EXPLib/datasets/Camera/toy_model_v1_inference/dev/images',
    "annotations": './EXPLib/datasets/Camera/toy_model_v1_inference/dev/satellite_objdet_coco_dataset.json',
    "num_epochs": 70,
    "num_workers": 0,
    "print_freq": 50
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    params["dataset_path"], params["annotations"],
    batch_size=params["batch_size"], num_workers=params["num_workers"]
)

# --- Model ---
model = create_ssdlite(num_classes=2, device=device)
optimizer = SGD([p for p in model.parameters() if p.requires_grad],
                lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)
scaler = GradScaler('cuda') if torch.cuda.is_available() else None

# --- Constraints ---
bbox_constraints = compute_bbox_constraints_from_coco_json(params["annotations"])

# --- Training ---
for epoch in range(params["num_epochs"]):
    train_one_epoch(model, optimizer, train_loader, device, epoch,
                    print_freq=params["print_freq"], scaler=scaler,
                    bbox_constraints=bbox_constraints)
    scheduler.step()

    if (epoch + 1) % 10 == 0:
        ckpt = f"ssd_constrainted_model_epoch_{epoch+1}.pth"
        torch.save(model.state_dict(), ckpt)
        print(f"Saved checkpoint: {ckpt}")

torch.save(model.state_dict(), "ssd_constrainted_model_final.pth")
print("Training finished and model saved.")